# Functional Programming in Python
 
This notebook introduces core functional programming (FP) concepts in Python.  
Python is in its core an object-oriented programming (OOP) language, not a FP language.  
In fact, almost everything in Python is an object and thus has methods and attributes.  
Still, Python provides in its standard library important modules for FP, namely:  
* operator — Standard operators as functions, like `add(x,y)`  
* functools — Functions of functions  
* itertools — Functions to create iterators
We say Python is a multi-paradigm language. 

There are also a lot of third-party FP libraries, we will explore  
* Toolz  
* Pyrsistent.

The following are the main points discussed in this notebook:  
- Pure vs Impure Functions  
- Higher-Order Functions  
- Partial functions  
- Curried functions  
- Function Composition  
- Immutability  
- Recursion  
- Closures
 
Each concept is demonstrated with examples.

Let's start by importing the necessary libraries.  
They have been abbreviated for convenience.

In [ ]:
import functools as fT
import itertools as iT
import operator as op

import toolz as Z

ZC = Z.curried
ZiT = Z.itertoolz
ZfT = Z.functoolz
ZdT = Z.dicttoolz

import sys
import time

import numpy as np
import pyrsistent as pyrs

## Purity  
Purity is one of the core concepts of FP.  
A function is pure when it always returns the same output for the same input.  
It should also should not change anything else in the code.  
In other words, the function has no side effects and doesn't depend on an external state.  
Let us look at an example.

In [ ]:
def add_pure(a, b):
    return a + b

In [ ]:
num1 = 1
num2 = 2
num1, num2

In [ ]:
add_pure(num1, num2)

In [ ]:
num1, num2

As expected, the pure function doesn't change the input variables.  
Since add is a standard operation, a pure version of it is given by the *operator* module from the standard library.

In [ ]:
op.add(num1, num2)

Now, let's look at an impure function.

In [ ]:
additions_made = 0
num1 = 1
num2 = 2

In [ ]:
def add_impure_counter(a, b):
    msg = "Hello"
    global additions_made
    additions_made += 1
    return a + b

The `global` declaration tells the function `add_impure_counter` that `additions_made` belongs to the global scope, i.e. it is not limited to a specific scope, it can be used everywhere in the code.  
(Local variables, on the other hand, can only be used within their local scope. In the code above, the `msg` variable is defined within the function. Thus, `msg` can't be reached outside that function.)  
Let us have a look at the variables before applying the function.

In [ ]:
num1, num2, additions_made

In [ ]:
add_impure_counter(num1, num2)

In [ ]:
num1, num2, additions_made

As we see here, `additions_made` changed by applying the function.  
Thus, the function is impure.  
Another impure version of the add function is the following

In [ ]:
num1 = 1
num2 = 2
num1, num2

In [ ]:
def add_global_vars():
    return num1 + num2

In [ ]:
add_global_vars()

Since `num1` and `num2` are neither given as arguments to `add_global_vars` nor defined in the function, they have to be *global* variables.  
Thus, python functions can read global variables without the `global` declaration.  
It was mandatory for us in the earlier example to declare the global variable, since we needed to assign to it a new value.  
Now, let us assign an other value to `num1`. 

In [ ]:
num1 = 2
add_global_vars()

As we can see now, running the same function `add_impure_ab()` gives us different results.  
Of course, we don't want this in FP.  
In other words, FP requires avoiding global variables.  
That is, we want to be as explicit as possible in FP by giving the function all the data it needs as input arguments.

Using global variables, we could make another impure version of the add function.

In [ ]:
result_add = 0
num1 = 1
num2 = 2

In [ ]:
def add_global_result(a, b):
    global result_add
    result_add += a
    result_add += b
    return result_add

In [ ]:
add_global_result(num1, num2)

In [ ]:
num1, num2, result_add

In [ ]:
add_global_result(num1, num2)

In [ ]:
num1, num2, result_add

Again, running `add_global_result(a, b)` twice leads to different results.  
Moreover, the first result is right but the second is wrong.  
This mistake pattern is also possible without functions as follows.

In [ ]:
result_add = 0
num1 = 1
num2 = 2

In [ ]:
for number in [num1, num2]:
    result_add += number

In [ ]:
result_add

In [ ]:
for number in [num1, num2]:
    result_add += number

In [ ]:
result_add

If such a thing happens, we say that we have forgotten to reinitialize the sum variable.  
The FP paradigm is there to prevent such mistakes, as they are very easy to make.  
Here, we can also see why functional programmers don't like loops, as they involve a state that has to be modified in each iteration.  
Now, it should have become clear how to find side effects in our algorithms and what their dangers are.  
Let us look at another example.  
Is the following function pure?

In [ ]:
def example_func_1(x):
    x = 2
    return x

Let us test it! (Experimenting is an awesome strategy for learning)

In [ ]:
num1 = 1
num1

In [ ]:
example_func_1(num1)

The output of the function is like we wanted.  
But is that enough to answer our question?  
No! We should look at the input variable too.

In [ ]:
num1

Now, we conclude that the function is pure, even though it doesn't look like it.  
The function here binds the value of the input variable `num1` to the name `x`.  
The value given here is immutable, as are all numbers in python.  
After assigning `2` to `x`, the name `x` is pointing to another immutable value.  
To understand this example, we should think about what it means to assign a value to a variable.  
Indeed, the assignment just binds the given value to the variable name.  
In fact, the memory address of the two values are totally different.  
To visualise this, we will use the `id` built-in function, which gives us the address of its input object in memory.

In [ ]:
def example_func_1_with_ids(x):
    print("Id of x:", id(x))
    x = 2
    print("Id of x:", id(x))
    return x

In [ ]:
example_func_1_with_ids(num1)

We can see here that the address of `x` changed after the assignment.  
Instead of modifying the input and thus causing a side effect, reassigning the local variable `x` just makes the variable point to another value.  
Let us now try a function similar to `example_func_1` but with a slight change.  
Instead of using an immutable input value, we will use a mutable one.

In [ ]:
number_list = [1]

In [ ]:
def example_func_2_with_ids(x):
    print("Id of x:", id(x),"Id of x[0]:", id(x[0]))
    x[0] = 2
    print("Id of x:", id(x),"Id of x[0]:", id(x[0]))
    return x

In [ ]:
example_func_2_with_ids(number_list)

In [ ]:
number_list

Using a mutable input allowed us to use its `__setitem__` method, as `x[0] = 2` is only syntactic sugar for `x.__setitem__(0, 2)`.  
This means that the name `x` is still bound to the same object after the assignment.  
Printing the values of `id(x)` shows us that the memory address of `x` didn't change after the assignment.  
Thus, the function modifies its input value.  
We have just found out why mutable types are avoided in FP.

## Variable reassignment  
As we have seen above, many problems are caused by assigning a new value to a variable after its original assignment.  
This means that, the same variable in Python might have different values in different times.  
From a pure functional programming standpoint, this is not appreciated.  
There, a variable should have the same value over the course of the program, as a variable is just a name that the value is bound on.  
The less changes happen in the code, the more transparent it is.  
Instead of reassigning a variable, we can just make a new variable instead.  
The new variable's name should simply point to how it differs from the other variable.  
This also means that generic variable names like `a` should be avoided, as they hold no meaning.  
Taking these things into consideration needs effort, but unless one is only working on a short one-use script, the effort will pay off.  

## Partial functions  
This concept is best understood with the `add` function above. This function expects two arguments.  
A partial function of the add function is a new add function with one argument fixed.  
That is, it only expects one argument.  
Let us make one by hand.

In [ ]:
def add1(num):
    return op.add(1, num)

In [ ]:
add1

In [ ]:
add1(1)

Instead of making the partial function manually, we will now use the `partial` function from the *functools* module.

In [ ]:
add1_partial = fT.partial(op.add, 1)
add1_partial

In [ ]:
add1_partial(1)

What would happen if we make a partial function of *add1_partial*?

In [ ]:
add_1_and_2_partial = fT.partial(add1_partial, 2)

In [ ]:
add_1_and_2_partial

In [ ]:
add_1_and_2_partial()

## Curried functions  
Say we are too lazy to make partial functions each time we need them.  
Curried functions are defined such that they can both be used as a partial function or receive all the needed arguments.  
The trick is to let functions expect only one argument at a time.  
Instead of the normal definition of the add function as a function with two arguments, we do the following.

In [ ]:
def curried_add(a):
    def inner(b):
        return a + b

    return inner

In [ ]:
curried_add

In [ ]:
curried_add(1)

In [ ]:
curried_add(1)(2)

Using the function is a bit different from the usual `f(1,2)`, but it brings a lot of advantages.  
Curried functions are especially useful when using higher-order functions, i.e. functions that depend on other functions.

The *toolz* library allows making curried function easily.

In [ ]:
@Z.curry
def curried_add_toolz(a, b):
    return a + b

In [ ]:
curried_add_toolz

In [ ]:
curried_add_toolz(1)

In [ ]:
curried_add_toolz(1)(2)

Note that it is also possible to run functions the usual way thanks to the `toolz.curry` trick.  
(`toolz.curry` is an example of a decorator.)

In [ ]:
curried_add_toolz(1, 2)

Of course, `curried_add(1, 2)` gives an error.

## The `map` function  
`map` is so important that it is one of the built-in functions.  
Map is a higher-order function allowing one to apply a function on an iterable, i.e. something we can iterate on, like lists for example.  
We talk about mapping a function on a list.  
But since lists in Python are mutable and tuples aren't, let us use tuples instead.  
(Using tuples like this isn't pythonic, as tuples are "typically used to store collections of heterogeneous data". This means that they shouldn't replace the use of `lists`. But following these ideals is not mandatory. Since immutability is crucial in FP, we will use tuples here as immutable lists.)  
Let us look at an example of the `map` function.  
We want to calculate the powers of two.

In [ ]:
tuple_0_to_5 = (0, 1, 2, 3, 4, 5)
tuple_0_to_5

In [ ]:
map(fT.partial(op.pow, 2), tuple_0_to_5)

Note, that we can't see the results since the python implementation of the map function returns an iterator.  
That means, we have to iterate over it to see the results.  
Let us first save the iterator in a variable.

In [ ]:
iter_powers_of_2 = map(fT.partial(op.pow, 2), tuple_0_to_5)

The `next` function allows us to explore the iterator one value at a time.

In [ ]:
next(iter_powers_of_2)

In [ ]:
next(iter_powers_of_2)

As you might have noticed, running the same command twice gives different results.  
In other words, `next` is not a pure function.  
### Iterators  
Iterators are objects that have 
* the `__next__` method, which is run when we apply the `next` function to the iterator and
* the `__iter__` method, which is run when we apply the `iter` function and returns the iterator object.

As we see here, the `__next__` method changes the state of the iterator.  
Getting an element with the `next` function means that it left the iterator.  
Once it leaves it, it can't go back to it.  
Thus, iterators are single-use objects.

Another impure method for exploring an iterator would to be use a `for` loop.

In [ ]:
for power_of_two in map(fT.partial(op.pow, 2), tuple_0_to_5):
    print(power_of_two)

In [ ]:
power_of_two

(As we see here, after the loop finishes, `power_of_two` still has the value of the last element in the iterator.)
Just like using the `next` function, the iterator will be empty after the loop, but that doesn't matter, since we didn't store it in a variable.  
The recommended way is to just wrap the map directly into the `tuple` type constructor, which builds a tuple with the same elements as the iterator with the same order.

In [ ]:
tuple(map(fT.partial(op.pow, 2), tuple_0_to_5))

One could also define his own map function that directly wraps the iterator in a tuple.
(The only benefit here is avoiding too many parentheses. )

In [ ]:
def tuple_map(function, iterable):
    return tuple(map(function, iterable))

In [ ]:
tuple_map(fT.partial(op.pow, 2), tuple_0_to_5)

However, it is important to consider, why the developers of Python chose to give an iterator as an output of the `map` function.  
### Making an iterator  
To dig deeper into this, let us implement an iterator object from scratch.  
It should give the integer numbers beginning with `start` until the last digit before `stop`.

In [ ]:
class Counter:
    def __init__(self, start, stop):
        self.current = start
        self.stop = stop

    def __iter__(self):
        return self

    def __next__(self):
        if self.current < self.stop:
            self.current += 1
            return self.current - 1
        raise StopIteration

To initialise the `Counter` object, we need a start and a stop value.  
The start value becomes the current value of the counter.  
In the `__iter__` method, we return `self`, that is the same instance.  
This method normally converts some object to an iterator, but since the object is already an iterator here, we just return the object itself.  
In the `__next__` method, we return the current value of the counter, in the case the counter is still less than the stop value.  
Otherwise, we raise the `StopIteration` exception.  
Let us initialize an instance of this class.

In [ ]:
Counter(0, 10**1000)

This iterator should iterate over all whole numbers until $10^{1000}$.  
This is a gigantic number!  
It makes no sense to convert the iterator to a tuple, as our memory is far from being enough to store all these numbers.  
This is why we say iterators are lazy, i.e. python could calculate the next elements of the iterator but it won't until they are needed.  
Let us calculate the powers of two with this iterator.

In [ ]:
iter_powers_of_2_counter = map(fT.partial(op.pow, 2), Counter(0, 10**1000))

Still, running the map function on the huge iterator works smoothly, since iterators only store a couple of data values at a time.  
To explore the huge iterator of powers, we'll use the `islice` function from the *itertools* module, which allows to make an iterator containing the first 10 elements of the iterator.

In [ ]:
tuple(iT.islice(iter_powers_of_2_counter, 10))

Of course, the values extracted by the function aren't in the input iterator anymore.  
## Encapsulation  
We should keep in mind, that `iter_powers_of_2_counter` is a variable.  
That is, we can change the state of the iterator at will.  
Rerunning the command above, for example, gives us the next 10 powers of two.  
To avoid this impurity, we will wrap this mutable state like the following.

In [ ]:
def tuple_powers_of_2(N):
    iter_powers_of_2_counter = map(fT.partial(op.pow, 2), Counter(0, 10**1000))
    return tuple(iT.islice(iter_powers_of_2_counter, N))

In [ ]:
tuple_powers_of_2(14)

This strategy is called encapsulation and is quite useful for dealing with mutability in Python.  
Encapsulation is the mechanism that groups related data and behaviour together and controls external interaction.  
In OOP, we often want to limit the interaction to selected methods.  
In pure FP, of course, we like immutability, i.e. we don't want any external modification of local mutable objects.  
Since the mutable state is stored in a local variable of the function and is not returned by the function, the only changes that happen to it are the ones listed in the function definition.  
This makes dealing with the problems of mutability simpler, as all the mutation is done in one place.  
Rerunning the function gives actually the same result, so the function is pure.

## Generators  
Generators are a kind of iterators that can be implemented very simply.  
One just has to define a function that instead of `return`ing values `yield`s them.  
Here is one for generating the powers of two.  
As usual, we will wrap it in a function for purity reasons.

In [ ]:
def tuple_powers_of_2_generated(N):
    def gen_powers_of_2():
        i = 0
        while True:
            yield 2**i
            i += 1

    generator = gen_powers_of_2()
    return tuple(iT.islice(generator, N))

In [ ]:
tuple_powers_of_2_generated(14)

The generator is defined by `gen_powers_of_2`.  
Since a generator is by definition an iterator, the `islice` function extracts the `N` needed elements one by one, as if we run the `next` function `N` times.  
Each time the `next` function is run, we receive the next value in the `yield` statement.  
It should have become obvious now, that unless we need an iterator with custom methods, generators are the way to go.  
### Generator expressions  
Making generators gets even easier with generator expressions.  
To make these, we use parenthesis like the following.

In [ ]:
(2**i for i in range(14))

As we can see, this command created a generator.  
The awesome thing about a generator expression is its simplicity, it looks like an english sentence.  
As usual, we use the `tuple` function to extract the results.

In [ ]:
tuple((2**i for i in range(14)))

## `lambda` expressions  
Lambda expressions provide an easy way to make anonymous functions, i.e. functions without a name.

In [ ]:
(lambda x: x**2)

In [ ]:
(lambda x: x**2)(2)

Without these, one would write the following code.  
```python  
def squared(x):  
  return x**2  
squared(2)  
```  
It should be clear now, how much time they can spare.  
Moreover, they spare us the mental effort of choosing a function name.  
Furthermore, using them prevents having too many variable names in the global scope.  
Anonymous functions play a keyrole in FP, especially in higher-order functions.  
One important aspect that we can see in `lambda` expressions is that they can be dealt with just like we deal with an integer or other data types.  
For example, they can be stored in a variable as in

In [ ]:
squared_lambda = lambda x: x**2
squared_lambda

We say Python has first class functions, i.e. it treats them as first class citizens.  
This also means that functions can be fed as arguments to other functions as in the `map` examples.  
Furthermore, functions can be given as an output as in the `functools.partial` examples.

## The `filter` function  
Filter is another built-in higher-order function. Like its name suggests, it allows to filter iterables.  
Filter is especially useful when combined with another function, i.e. this function will only be applied to the elements of the iterable that satisfy the filter condition.  
Like the `map` function, `filter` also expects a function and an iterable as an argument.

In [ ]:
tuple_0_to_10 = tuple(range(0, 11))
tuple_0_to_10

In [ ]:
filter(lambda x: x % 2 == 0, tuple_0_to_10)

It also returns an iterator as the output.

In [ ]:
tuple(filter(
    lambda x: x % 2 == 0,
    tuple_0_to_10
))

Filters are especially useful when combined with another function, i.e. this function will only be applied to the elements of the iterable that satisfy the filter condition.  
Let us multiply all the number that passed the filter by two.  
For this, we use a partial function of the multiplication function `mul` given by the *operator* module.

In [ ]:
tuple(map(
    fT.partial(op.mul, 2),
    filter(
        lambda x: x % 2 == 0,
        tuple_0_to_10
    )
))

As you see, it has become harder to visualize which function goes into which function.  
Here is where the role of `compose` comes into play.

## Composed functions  
The `compose` function from the *toolz.curried* package allows to compose two functions or more into one function.

In [ ]:
ZC.compose(
    tuple,
    ZC.map(fT.partial(op.mul, 2)),
    ZC.filter(lambda x: x % 2 == 0)
)

Note that we have also used the `map` and `filter` functions from the *toolz.curried* package, since these would normally require an iterable as a second argument.  
But thanks to the curried implementations of these functions, the first argument is enough.  
Let us now use the composed function.

In [ ]:
ZC.compose(
    tuple,
    ZC.map(fT.partial(op.mul, 2)),
    ZC.filter(lambda x: x % 2 == 0)
)(
    tuple_0_to_10
)

The idea is that the last function given to the `compose` is the first one that is run on the input.  
That is, the numbers are filtered first, then each one of them is multiplied by two.  
Here is a sketch of what is happening:  
```
(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10)
    |
    |             ZC.filter(lambda x: x % 2 == 0)
    v
0,    2,  4,  6,  8,  10   (as an iterator)
    |
    |             ZC.map(fT.partial(op.mul, 2))
    v
0,    4,  8,  12, 16, 20   (as an iterator)
    │
    │             tuple
    v
(0,    4, 8,  12, 16, 20)
```

## The `reduce` function  
The reduce function goes through each element of a given iterable and adds it in a selected manner to the accumulated value.  
`reduce` is implemented in the *functools* module.  
Let us look at the simple example of summing an iterable.

In [ ]:
fT.reduce(op.add, tuple_0_to_10)

This example was two easy, as we didn't make the function.  
`reduce` expects a function with two necessary arguments:  
1. `acc`: it stands for the accumulated value  
2. `y`: it is the current element of the iterable that should be added  
Besides the function, `reduce` expects the iterable and an optional `initial` argument, which should be given the initial value of the `acc` argument.  
Otherwise the first element of the iterable is used.  
(Of course, the first element doesn't get accumulated twice.)  
To make these clearer, let us implement the last command by hand.

In [ ]:
fT.reduce(
    lambda acc, y: acc + y,
    tuple_0_to_10,
    0
)

But the `add` function from the *operator* module does exactly the same thing.  
Also, we didn't need to give the initial value of 0, since it doesn't change anything.

Let us work on a more complicated example.  
Now, we want to manually reduce the elements of a `range` object to a tuple.

In [ ]:
fT.reduce(
    lambda acc, y: acc + (y,),
    range(0, 11),
    ()
)

As you see, we concatenate the accumulated tuple with a new tuple containing the value to be accumulated. (We need a comma within the parentheses to differentiate the tuple from just mathematical parentheses surrounding the element `y`.)  
Without the initial empty tuple, the program gives an error.  
Following is an example of multiplying all the numbers in an iterable together.

In [ ]:
fT.reduce(op.mul, tuple_0_to_10)

Of course, the result is 0, since the first element is 0.
### Example of combining `map` and `reduce` 
This example has been used to explain MapReduce in the course.  
We would like to calculate how often a word comes in a paragraph.  
So we will reduce the text to a dictionary of words as keys and their frequencies as values.  
However, we first have to format the words, so that they are all lower case and that punctuation is removed from them.  
Following is the example paragraph we will use.  

In [ ]:
paragraph = "Functional programming treats coding like a math problem where data stays the same and functions do all the heavy lifting. Instead of telling the computer exactly how to spin through a loop, you use tools like map and reduce to transform entire lists at once. Every function is pure, which means it will always give the same answer if you provide the same input, making the logic very predictable. While it feels strange at first to avoid changing variables or using standard loops, this approach makes your work much cleaner and easier to test. It is like building with solid blocks that never break or change shape unexpectedly. Because the data never changes, this style is a perfect fit for modern systems where many different parts of a program need to run at the exact same time without crashing into each other."

Next, we will define the formatter function.   

In [ ]:
def formatter(a):
    if not a[0].isalpha():
        a = a[1:]
    if not a[-1].isalpha():
        a = a[:-1]
    return a.lower()

Now, we will define the function used for reducing the dictionary.  
If the word is already in the dictionary, it increments the value.  
Otherwise, a new key is defined with the value 1.  

def accumulate_dict(acc, item):
    if (item in acc):
        return {**acc, item: acc[item] + 1}
    return {**acc, **{item: 1}}

One could actually make an equivalent lambda function in one line only.  

In [ ]:
lambda acc, item: {**acc, item: acc[item] + 1} if (item in acc) else {**acc, **{item: 1}}

Next, we will combine everything to make the dictionary.  
Note, that we use the `split()` string method, which parses the paragraph to a list of words.  

In [ ]:
dict_word_frequency = fT.reduce(
    lambda acc, item: {**acc, item: acc[item] + 1} if (item in acc) else {**acc, **{item: 1}},
    map(
        formatter,
        paragraph.split()
    ),
    {}
)
print(dict_word_frequency)

As you can see, the dictionary is not well sorted.  
So, we will use the `sorted` built-in function to easily find out which words were the most frequent.  
For this purpose, we need to apply the `items()` method on the dictionary so that we get `(key, value)` pairs of all its elements.  
We give the `key` parameter a function which for each `(key, value)` pair returns the `value` so that the sorting is based on the number of occurrences.  
We only print the first 10 pairs.

In [ ]:
sorted(
    dict_word_frequency.items(),
    key=lambda item: item[1],
    reverse=True
)[:10]

The same functions could be used on a whole book to get more meaningful insights.  
## Closures  
Closures are a set of variables that get carried along with a function, even though they are neither part of its arguments nor its body.  
Let us look at an example:

In [ ]:
def closure_maker():
    closure_variable = 1

    def inner():
        return closure_variable

    return inner

In [ ]:
closure_carrier = closure_maker()
closure_carrier

In [ ]:
closure_carrier()

Here, the variable `closure_variable` travels along with the function `inner`, that is assigned to the variable `closure_carrier` even though it's in the local scope of the function `closure_maker`.  
This is called capturing the environment.  
We are only able to do this thanks to first class functions.  
Closures could be made such that they can be mutated outside the function.  
This is an impure functional strategy which may remind us of encapsulation in OOP.  
Here, the programmer chooses also exactly how he wants the closure to be changed outside the function.  
Let us look at a closure implementation of a counter.

In [ ]:
def counter_closure_maker():
    counter = 0
    def inner():
        nonlocal counter
        counter += 1
        return counter
    
    return inner

In [ ]:
counter_closure_carrier = counter_closure_maker()

In [ ]:
counter_closure_carrier()

Each time `counter_closure_carrier` is run, the variable `counter` gets incremented.  
The `nonlocal` statement is needed since the `inner` function changes a variable outside its local scope.  
This is still much better than using the `global` statement, since running `counter_closure_carrier` is the only possible interaction with the variable `counter` outside the definition of `counter_closure_maker`.  
One could again wrap the closure code in another function to avoid unwanted side effects.  
## Recursion
Another essential strategy used in FP is recursion.  
In simple words, a recursive function is a function that uses itself within its own definition.  
Of course, the function must also have a base case to stop the recursion, otherwise using the function would cause an infinite loop.  
Let us look at a simple recursive function that sums the elements of a.

In [ ]:
def sum_recursive(numbers):
    if len(numbers) == 0:
        return 0
    return numbers[0] + sum_recursive(numbers[1:])

The base case here is when the tuple is empty, then the function returns 0.  
In the recursive case, the function returns the sum of the first element with the function applied on the other elements.  
However, Python has a big constrain on recursion.  
Recursions in Python have a limited depth (3000 in my computer)  
You can find out by running the following function from the *sys* module.

In [ ]:
sys.getrecursionlimit()

In our example, this means that the tuple to be summed isn't allowed to have more than 3000 elements.

In [ ]:
sum_recursive((1, ) * 100)

However, running this causes a `RecursionError`.  
```python  
sum_recursive((1, ) * 10000)  
```  
Python also allows to reset the recursion limit with `sys.setrecursionlimit(limit)`.  
Still, one should avoid deep recursions in Python and use alternative strategies.  
(In the case of the sum function, using `reduce` is probably the best strategy.)

## Memoize  
Another cool trick allowed by the *toolz.functools* package is memoization.  
This means, that the results of the function get saved.  
Of course, this is only interesting for pure functions, otherwise the function could be expected to give another result for the same input on a later stage.  
To define a function with memoization, we add the `memoize` decorator from this package.  
(The *functools* module provide a similar decorator called `lru_cache`, you can lookup the documentation for more details.)  
Let us start by a simple example.

In [ ]:
@ZfT.memoize
def foo(x):
    time.sleep(10)
    return x + 3

In [ ]:
foo(3)

In [ ]:
foo(3)

Look at how one has to wait 10 seconds for the first result, while the second result is immediately returned.  
A typical recursion example is the Fibonacci Sequence, which is defined like the following.

In [ ]:
def fibonacci_recursive_normal(n):
    if n == 1 or n == 2:
        return 1
    return fibonacci_recursive_normal(n - 2) + fibonacci_recursive_normal(n - 1)

This is a very bad use of recursion.  
Let us make a verbose version of the function to visualize the problem.

In [ ]:
def fibonacci_recursive_verbose(n, level = 0):
    print(level * '\t' + f'(n = {n}')
    if n == 1 or n == 2:
        print(level * '\t' + f'n = {n})')
        return 1
    a = fibonacci_recursive_verbose(n - 2, level + 1)
    b = fibonacci_recursive_verbose(n - 1, level + 1)
    print(level * '\t' + f'n = {n})')
    return  a + b 

In [ ]:
fibonacci_recursive_verbose(5)

A parenthesis is opened each time the function is run.  
It is then closed directly before returning the result.   
We store recursion depth in the `level` variable.  
Print indentation is based on recursion depth to visualize call hierarchy.  
It should have become clear that this version of the function calculates the same values many times, which makes it inefficient.  
Now, let us save the execution time to compare this implementation with the memoized one.  

In [ ]:
fibonacci_test_number = 40

In [ ]:
time_start = time.time()
print(fibonacci_recursive_normal(fibonacci_test_number))
time_end = time.time()
timed_fibonacci_recursive_normal = time_end - time_start

Let us define the memoized version of the function.  

In [ ]:
@ZfT.memoize
def fibonacci_recursive_memoized(n):
    if n == 1 or n == 2:
        return 1
    return fibonacci_recursive_memoized(n - 2) + fibonacci_recursive_memoized(n - 1)

In [ ]:
time_start = time.time()
print(fibonacci_recursive_memoized(fibonacci_test_number))
time_end = time.time()
timed_fibonacci_recursive_memoized = time_end - time_start

In [ ]:
print(f"Elapsed time:")
print(f"Normal Fibonacci: {timed_fibonacci_recursive_normal}")
print(f"Memoized Fibonacci: {timed_fibonacci_recursive_memoized:.3e}")
print(f"Ratio Normal / Memoized: {timed_fibonacci_recursive_normal / timed_fibonacci_recursive_memoized:.3e}")

Even with such a small number, the benefits of memoization become quite clear.  
Let us now try a higher number.

In [ ]:
fibonacci_recursive_memoized(950)

If we use the normal Fibonacci implementation with this number, it would give a RecursionError.  
One should mention though that memoization comes with the cost of filling memory.  
Since we have calculated the 950th Fibonacci element, we have stored 950 numbers in memory.  
Thus, one should always figure out whether memoization suits the chosen solution to the problem or not before using it.
## Tail recursion
Recursion could also look a bit different.  
We could optimize the recursion scheme of the normal Fibonacci implementation with tail recursion.  
To understand this concept, let us first return to the recursive sum example.  

In [ ]:
def sum_recursive(numbers):
    if len(numbers) == 0:
        return 0
    return numbers[0] + sum_recursive(numbers[1:])

Let us run it on the following tuple `(0, 1, 2, 3, 4, 5)`.  

In [ ]:
sum_recursive((0, 1, 2, 3, 4, 5))

What happens behind the scenes is:
```
sum_recursive((0, 1, 2, 3, 4, 5))
=> 0 + sum_recursive((1, 2, 3, 4, 5))
=> 0 + (1 + sum_recursive((2, 3, 4, 5)))
=> 0 + (1 + (2 + sum_recursive((3, 4, 5))))
=> 0 + (1 + (2 + (3 + sum_recursive((4, 5)))))
=> 0 + (1 + (2 + (3 + (4 + sum_recursive((5, ))))))
=> 0 + (1 + (2 + (3 + (4 + (5 + sum_recursive(()))))))
=> 0 + (1 + (2 + (3 + (4 + (5 + 0)))))
=> 0 + (1 + (2 + (3 + (4 + 5))))
=> 0 + (1 + (2 + (3 + 9)))
=> 0 + (1 + (2 + 12))
=> 0 + (1 + 14)
=> 0 + 15
=> 15
```
It should have become clear that this is not the optimal recursive function for summing elements.  
It would be more efficient to directly accumulate the elements instead of collecting them separately in the memory. (Think about a long tuple...)  
If we simultaneously accumulate the result, the last operation performed when running the function will always be a recursive call of the function.  
This is why it is called *tail recursion*.  
To implement it, we accumulate the result in a second argument. 

In [ ]:
def sum_tail_recursive(numbers):
    def inner(numbers, sum):
        if len(numbers) == 0:
            return sum
        return  inner(numbers[1:], sum + numbers[0])
    
    return inner(numbers, 0)

In [ ]:
sum_tail_recursive((0, 1, 2, 3, 4, 5))

Now, it is much simpler.  
```
sum_tail_recursive((0, 1, 2, 3, 4, 5))
=> sum_tail_recursive((1, 2, 3, 4, 5), 0 + 0)
=> sum_tail_recursive((2, 3, 4, 5), 0 + 1)
=> sum_tail_recursive((3, 4, 5), 1 + 2)
=> sum_tail_recursive((4, 5), 3 + 3)
=> sum_tail_recursive((5, ), 6 + 4)
=> sum_tail_recursive((), 10 + 5)
=> 15
```
Then, let us do the same strategy with the recursive Fibonacci function.  

In [ ]:
def fibonacci_tail_recursive(n):
    if n == 0:
        return 0
    def inner(n, a, b):
        if n == 1:
            return b
        return inner(n - 1, b, a + b)
    
    return inner(n, 0, 1)

In [ ]:
fibonacci_tail_recursive(950) == fibonacci_recursive_memoized(950)

Tail recursion made the fibonacci function much more effective, compared with `fibonacci_recursive_verbose`.  
In this case, tail recursion allowed us to reach much higher numbers, as my computer is still having problems calculating the 50th Fibonacci number with `fibonacci_recursive_normal` since the recursion depth increases exponentially with two recursive calls per function invocation.  
Still, tail recursion doesn't prevent the RecursionError, it just makes the recursion limit harder to reach.  
There are many programming languages that offer tail-call optimization, which allow tail-recursion to only use a constant memory space.  
In such languages, tail recursion doesn't have a limit.  
## Replacing recursion by iterative solutions
A standard strategy for avoiding the recursion limit in Python is to make an iterative solution, which is impure by default.  
Again, we will wrap the solution to avoid unwanted side effects.  
The most straightforward way to do this is to take the tail-recursive version and transform it into a loop, where the recursive state updates are replaced by iterative updates to local variables.  

In [ ]:
def fibonacci_iterative(n):
    if n == 0:
        return 0
    a, b = 0, 1
    while True:
        if n == 1:
            return b
        n, a, b = n - 1, b, a + b

Note, that we change the values of all the state variables at once.  
This trick is called tuple unpacking, as `n, a, b` is actually a tuple.  
The right-hand side is evaluated before any assignment occurs, ensuring that the new values are based on the *old* values of `a` and `b`.  
Since the original function is tail-recursive, only one state transition is needed per iteration.  

In [ ]:
fibonacci_iterative(950)

In [ ]:
fibonacci_iterative(950) == fibonacci_recursive_memoized(950)

As we can see, the iterative solution gives the same result as the memoized one.  

In [ ]:
time_start = time.time()
print(fibonacci_iterative(fibonacci_test_number))
time_end = time.time()
timed_fibonacci_iterative = time_end - time_start

In [ ]:
print(f"Elapsed time:")
print(f"Recursive Fibonacci: {timed_fibonacci_recursive_normal}")
print(f"Memoized Fibonacci: {timed_fibonacci_recursive_memoized:.3e}")
print(f"Iterative Fibonacci: {timed_fibonacci_iterative:.3e}")
print(f"Ratio Normal / Memoized: {timed_fibonacci_recursive_normal / timed_fibonacci_recursive_memoized:.3e}")
print(f"Ratio Normal / Iterative: {timed_fibonacci_recursive_normal / timed_fibonacci_iterative:.3e}")

Now, we can calculate elements in the Fibonacci sequence that are much higher than what the recursion limit can afford.

In [ ]:
fibonacci_iterative(15000)

For much higher numbers, the Fibonacci element gets calculated but it can't be printed, as it exceeds the default limit of string digits that an integer is allowed to convert into.  

## Pyrsistent  
Let us now have a look at another library called Pyrsistent.  
The goal of this library is to offer alternative python objects similar to those we are used to, but that are immutable.  
Python lists, dictionaries and sets are mutable by default, that is, they can be modified.  
Pyrsistent PVectors (equivalent to lists), PMaps (equivalent to dictionaries) and PSets (equivalent to sets) can't be changed.  
Now, python objects can be edited with methods.  
Of course, this is not allowed with Pyrsistent objects.  
The same methods can be used Pyrsistent objects, but they make a new object with the wanted change instead.

### PVectors  
Vectors can be defined with both `v` and `pvector`.

In [ ]:
pyrs.pvector([1, 2, 3]) == pyrs.v(1, 2, 3)

As we can see the two methods are equivalent.  

In [ ]:
pvector1 = pyrs.v(3, 4, 10, 14)
pvector1

In [ ]:
pvector1.append(1000)

In [ ]:
pvector1

`pvector1` didn't change after appending 1000.  
The `set` method allows changing the value of the element in some index.

In [ ]:
pvector1.set(0, 5000)

In [ ]:
pvector1

Again, the vector stayed untouched.

### PMaps
PMaps can be defined with both `m` and `pmap`.

In [ ]:
pmap1 = pyrs.m(nano = -9, micro = -6)
pmap1

In [ ]:
pmap1 == pyrs.pmap({"nano": -9, "micro": -6})

In [ ]:
pmap1.set("gega", 9)

In [ ]:
pmap1

Similarly, the PMap stayed untouched.
### `freeze` and `thaw`  
These two methods allow us convert normal python objects to Pyrsistent objects or the other way around.

In [ ]:
complex_dict = {"sublist": [["a", "b", "c"], ["d", "f"]]}
complex_dict

In [ ]:
pyrs.freeze(complex_dict)

The result is now composed of Pyrsistent objects in all levels.  
The input didn't change.  

In [ ]:
complex_dict

`thaw` does the inverse effect.  
Thus, we can use it to undo the freezing.  

In [ ]:
pyrs.thaw(pyrs.freeze(complex_dict))

Like expected, it's equal to the input.

## Performance
Last but not least, the goal behind FP programming is not to improve performance.  
Respecting the FP rules makes sure to avoid many type of errors by default and thus think more about what actually matters.  
This has often the *side effect* of a worse performance.  
For example, when an array is immutable, one needs to make a new array instance for each change, instead of modifying it in place.  
Since there is no Tail Call Optimization in Python, recursion is always slower than iterative approaches.  
It is the programmer in the end who decides whether the performance gain is worth it, as it opens the door to many errors.
On the other hand, using only pure functions allows memoization, which could make the program much faster. (Still, it comes with the cost of filling memory.)  
Moreover, if all the used functions are pure, then they are by default ready to be run in parallel, which could also make the program perform better.  

To sum up, we have learned in this notebook the core strategies used in FP.  
Feel free to add the strategies that you liked the most to your workflow.